# Week 3 — ETAS Baseline Model
FaultLine: fits a real spatio-temporal ETAS model to San Andreas seismicity using
the `etas` library (Mizrahi et al., ETH Zurich / SED), then forward-simulates
30-day forecasts and aggregates them onto our 10 dissolved fault sections from
Week 2.

**Why this library instead of a from-scratch implementation:** a direct joint
MLE fit of ETAS parameters was tested and found to suffer from a real,
documented identifiability problem (K, c, and p trade off against each other
on a flat likelihood ridge). This library uses the EM-based stochastic
declustering algorithm (Zhuang, Ogata & Vere-Jones, 2002), which sidesteps that
exact issue -- confirmed by running its own validated example end-to-end before
adapting it to our data.

**Setup required first (run once in a terminal, not in this notebook):**
```
pip install git+https://github.com/lmizrahi/etas
pip install seismostats   # undeclared dependency of etas.simulation
```


In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd

from etas.inversion import ETASParameterCalculation
from etas.simulation import ETASSimulation

pd.set_option("display.max_columns", None)


## Configuration

Design decisions locked in during planning:
- **Magnitude threshold (M4.0+):** defines "the event" FaultLine forecasts a
  30-day probability for.
- **Train/validate/test split:** train 2016-2021 (includes the 2019 Ridgecrest
  sequence -- valuable aftershock-triggering signal for ETAS to learn from),
  validate 2022, test 2023-2025.
- **Auxiliary period:** ETAS needs a history-only buffer before the scored
  training window, so events just before `timewindow_start` aren't penalized
  as "unexplained" triggers. Our catalog only starts in 2016, so this buffer
  is shorter than ideal (2 years, vs. ~10 years in the library's own
  California example) -- a real, honest limitation, not a bug.
- **theta_0:** initial parameter guess. Reused directly from the library's own
  validated California-catalog example config, since it's a well-tested
  starting point for this exact region and gives the EM algorithm a head
  start (it does not bias the final answer, only convergence speed).

In [2]:
MAG_THRESHOLD = 4.0        # the event whose 30-day probability we're forecasting
MC = 2.0                   # completeness magnitude (from Week 1 Gutenberg-Richter analysis)
DELTA_M = 0.1               # magnitude bin size

AUXILIARY_START = "2016-01-01 00:00:00"   # earliest data we have -- true auxiliary
                                            # history would ideally be longer
TIMEWINDOW_START = "2018-01-01 00:00:00"   # scored training window starts here
TIMEWINDOW_END = "2021-12-31 00:00:00"     # end of training (matches project split)
TESTWINDOW_END = "2025-12-31 00:00:00"     # reserved for later evaluation

COPPERSMITH_MULTIPLIER = 100   # matches the library's own validated example;
                                 # controls how far apart two events can be
                                 # before being treated as uncorrelated
REGION_BUFFER_KM = 20           # same buffer used for the Week 2 earthquake-to-
                                 # segment spatial join, for consistency

THETA_0 = {
    "log10_mu": -5.8, "log10_k0": -2.6, "a": 1.8, "log10_c": -2.5,
    "omega": -0.02, "log10_tau": 3.5, "log10_d": -0.85, "gamma": 1.3, "rho": 0.66,
}

N_SIMULATIONS = 1000   # Monte Carlo forecast realizations


## Load Week 2 outputs

We need the dissolved 10-section fault geometry and the earthquakes already
assigned to it (`quakes_assigned`), rather than the raw statewide catalog --
`quakes_assigned` is already filtered to events within our region of interest
(the same 20 km buffer we'll use to build the ETAS region polygon below), so
this becomes exactly the regional catalog ETAS should be fit to.

In [3]:
faults = gpd.read_file("../data/raw/san_andreas_faults.geojson")
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

quakes = pd.read_parquet("../data/raw/usgs_2016_2026.parquet")
quakes = quakes.rename(columns={"magnitud": "mag"})
quakes["time"] = pd.to_datetime(quakes["time"]).dt.tz_localize(None)

# Re-run the spatial join from Week 2 so quakes_assigned reflects the current
# faults geometry (fault_segment_id is regenerated from row index each time
# faults is reloaded/dissolved, so this must be redone rather than cached).
PROJECTED_CRS = "EPSG:3310"
MAX_ASSIGN_DIST_KM = 20

quake_points_m = gpd.GeoDataFrame(
    quakes.copy(), geometry=gpd.points_from_xy(quakes["longitude"], quakes["latitude"]), crs="EPSG:4326"
).to_crs(PROJECTED_CRS)
faults_m = faults.to_crs(PROJECTED_CRS)

joined = gpd.sjoin_nearest(
    quake_points_m, faults_m[["fault_segment_id", "geometry"]],
    max_distance=MAX_ASSIGN_DIST_KM * 1000, distance_col="dist_to_fault_m",
)
quakes_assigned = joined.drop(columns="geometry").rename(columns={"time": "event_time"})
quakes_assigned["dist_to_fault_km"] = quakes_assigned["dist_to_fault_m"] / 1000

print(f"Fault sections: {len(faults)}")
print(f"Earthquakes assigned to the region: {len(quakes_assigned):,}")


Fault sections: 10
Earthquakes assigned to the region: 12,030


## Build the ETAS input catalog

The library requires exactly these columns: `id, latitude, longitude, time, magnitude`.
This is a direct subset/rename of what we already have -- no new data needed.

In [4]:
etas_catalog = quakes_assigned.rename(columns={
    "event_time": "time",
    "event_id": "id",
})[["id", "latitude", "longitude", "time", "magnitude"]].copy()

print(etas_catalog.shape)
etas_catalog.head()


(12030, 5)


,id,latitude,longitude,time,magnitude
35,ci37509616,34.157167,-117.469667,2016-01-02 07:50:35.200,1.81
45,ci37509720,34.059000,-117.267833,2016-01-02 18:02:32.490,1.52
49,nc72572935,36.675333,-121.304667,2016-01-03 01:49:09.860,2.10
59,ci37509920,35.349667,-119.871667,2016-01-03 10:06:13.350,1.60
66,nc72573175,36.562167,-121.165167,2016-01-03 15:59:18.200,1.81


## Build the region polygon (`shape_coords`)

The library needs a boundary polygon defining the region ETAS is fit to --
background rate (`mu`) is estimated per unit area of this region, and it
determines which events count as "targets" the model is scored against.

We build this by buffering our dissolved fault geometry outward by the same
20 km used for the Week 2 earthquake-to-segment assignment (for consistency),
then taking its convex hull for a single clean boundary. Note the coordinate
order: the library expects **[latitude, longitude]** pairs, the reverse of the
usual (lon, lat) GeoJSON/shapely convention -- easy to get backwards silently,
so this is called out explicitly rather than left implicit.

In [8]:
def build_shape_coords(faults_wgs84, buffer_km, projected_crs):
    """Boundary polygon around the fault geometry, in the etas library's
    expected [[lat, lon], ...] format (note: lat/lon order, not lon/lat)."""
    faults_proj = faults_wgs84.to_crs(projected_crs)
    # Take the convex hull FIRST (cheap -- just a set of points), then buffer
    # that single simple polygon once. Buffering before hulling forces shapely
    # to buffer every individual line fragment (thousands of small disconnected
    # pieces from the original 8,042-fragment fault file) and then union all of
    # them together, which is dramatically more expensive than buffering one
    # already-simple convex shape.
    hull = faults_proj.geometry.union_all().convex_hull
    buffered_hull = hull.buffer(buffer_km * 1000)
    hull_wgs84 = gpd.GeoSeries([buffered_hull], crs=projected_crs).to_crs("EPSG:4326").iloc[0]
    lons, lats = hull_wgs84.exterior.coords.xy
    return [[lat, lon] for lat, lon in zip(lats, lons)]

shape_coords = build_shape_coords(faults, REGION_BUFFER_KM, PROJECTED_CRS)
print(f"Region polygon has {len(shape_coords)} vertices")

Region polygon has 95 vertices


## Fit the ETAS model

This runs the EM-based declustering inversion. Expect this to take a while and
print detailed iteration-by-iteration logs (productivity, background rate
estimates, convergence). Watch for the same signal we validated earlier: the
"difference to previous" value should shrink smoothly toward zero, and final
parameters should land on interior values, not pinned at a range boundary.

In [9]:
print(f"shape_coords defined: {len(shape_coords)} vertices")

shape_coords defined: 95 vertices


In [10]:
metadata = {
    "name": "FaultLine ETAS Baseline",
    "id": "faultline_v1",
    "catalog": etas_catalog,
    "auxiliary_start": AUXILIARY_START,
    "timewindow_start": TIMEWINDOW_START,
    "timewindow_end": TIMEWINDOW_END,
    "testwindow_end": TESTWINDOW_END,
    "theta_0": THETA_0,
    "mc": MC,
    "delta_m": DELTA_M,
    "coppersmith_multiplier": COPPERSMITH_MULTIPLIER,
    "shape_coords": shape_coords,
}

calc = ETASParameterCalculation(metadata)
calc.prepare()
fitted_params = calc.invert()

import os
os.makedirs("../data/processed/etas_output", exist_ok=True)
calc.store_results("../data/processed/etas_output/", store_pij=True)

print("\nFitted parameters:")
print(fitted_params)



Fitted parameters:
{'log10_mu': np.float64(-6.248023407255552), 'log10_iota': None, 'log10_k0': np.float64(-2.8111484391620785), 'a': np.float64(1.0137566367594883), 'log10_c': np.float64(-4.647679379730899), 'omega': np.float64(-0.23763176771688221), 'log10_tau': np.float64(3.1097615487767127), 'log10_d': np.float64(-1.0165862212406518), 'gamma': np.float64(0.3953084442178622), 'rho': np.float64(0.5521807608527571)}


**Before trusting this fit:** check that convergence looked like the smooth,
monotonic pattern from our validation run, and that no parameter landed
exactly on one of its bounds (`log10_mu_range`, `a_range`, etc., saved in the
output JSON) -- a boundary-pinned parameter is the same degeneracy warning
sign we saw when our own from-scratch attempt failed.

In [12]:
with open("../data/processed/etas_output/parameters_faultline_v1.json") as f:
    params_saved = json.load(f)

final = params_saved["final_parameters"]
print(f"n_target_events: {params_saved['n_target_events']}")
print(f"n_iterations: {params_saved['n_iterations']}")
print(f"n_hat (estimated triggered events): {params_saved['n_hat']:.1f}")
print()
for k, v in final.items():
    print(f"{k:12s}: {v}")


n_target_events: 2014
n_iterations: 15
n_hat (estimated triggered events): 113.1

log10_mu    : -6.248023407255552
log10_iota  : None
log10_k0    : -2.8111484391620785
a           : 1.0137566367594883
log10_c     : -4.647679379730899
omega       : -0.23763176771688221
log10_tau   : 3.1097615487767127
log10_d     : -1.0165862212406518
gamma       : 0.3953084442178622
rho         : 0.5521807608527571


## Simulate 30-day forecasts

Forward-simulate many possible 30-day continuations from the end of the
training window, using the fitted model. Each realization is one possible
future; the fraction of realizations containing a qualifying (M4.0+) event on
a given fault section is that section's forecast probability.

In [13]:
simulation = ETASSimulation(calc)
simulation.prepare()

forecast_df = simulation.simulate_to_df(
    forecast_n_days=30,
    n_simulations=N_SIMULATIONS,
    m_threshold=MAG_THRESHOLD,
)

print(f"Total simulated qualifying events across {N_SIMULATIONS} realizations: {len(forecast_df)}")
forecast_df.head()


Total simulated qualifying events across 1000 realizations: 365


,latitude,longitude,magnitude,time,catalog_id
id,,,,,
3061,33.185709,-115.645065,4.0,2022-01-12 14:02:56.643533851,3
3089,36.967571,-121.626759,4.0,2022-01-03 18:03:19.493836090,6
3075,36.582573,-121.163551,4.4,2022-01-13 22:44:26.005254674,10
3072,36.667425,-121.299757,4.5,2022-01-18 13:42:02.357377351,11
3048,33.267256,-115.692135,4.1,2022-01-09 02:06:55.123832582,18


## Aggregate simulated events onto the 10 fault sections

Same spatial-join pattern as Week 2, applied to the simulated events instead
of the real catalog. A section that never got a simulated event in any
realization must still appear at probability 0 -- not silently disappear from
the output, which is why we reindex against the full section list rather than
just grouping and stopping there.

In [14]:
def aggregate_forecast_to_segments(forecast_df, faults_wgs84, projected_crs, max_dist_km, n_simulations):
    """Returns one row per fault_segment_id: the fraction of simulations
    (out of n_simulations) that produced at least one qualifying event there."""
    events = gpd.GeoDataFrame(
        forecast_df.copy(),
        geometry=gpd.points_from_xy(forecast_df["longitude"], forecast_df["latitude"]),
        crs="EPSG:4326",
    ).to_crs(projected_crs)
    faults_proj = faults_wgs84.to_crs(projected_crs)

    joined = gpd.sjoin_nearest(
        events, faults_proj[["fault_segment_id", "geometry"]],
        max_distance=max_dist_km * 1000, distance_col="dist_m",
    )

    hits = joined[["fault_segment_id", "catalog_id"]].drop_duplicates()
    hit_counts = hits.groupby("fault_segment_id").size()

    all_segments = faults_wgs84["fault_segment_id"]
    probabilities = (hit_counts.reindex(all_segments, fill_value=0) / n_simulations).rename("probability_30d")
    return probabilities.reset_index()

etas_baseline_forecast = aggregate_forecast_to_segments(
    forecast_df, faults, PROJECTED_CRS, MAX_ASSIGN_DIST_KM, N_SIMULATIONS
)
etas_baseline_forecast = etas_baseline_forecast.merge(
    faults[["fault_segment_id", "section_na"]], on="fault_segment_id"
)
etas_baseline_forecast.sort_values("probability_30d", ascending=False)


,fault_segment_id,probability_30d,section_na
2,2,0.126,Creeping section
7,7,0.072,San Bernardino Mountains section
1,1,0.055,Coachella section
8,8,0.032,Santa Cruz Mountains section
5,5,0.012,Parkfield section
3,3,0.011,Mojave section
0,0,0.007,Cholame-Carrizo section
4,4,0.006,North Coast section
9,9,0.006,Shelter Cove Section
6,6,0.005,Peninsula section


**Worth checking against what we already know:** sections like Creeping
(dense, near-continuous seismicity) should show a meaningfully higher 30-day
probability than sections like Parkfield or Shelter Cove (chronically quiet,
100% NaN b-value in Week 2) -- if a chronically-quiet section comes back with
a suspiciously high probability, or a historically active one comes back
near zero, that's worth investigating before trusting this baseline.

In [17]:
forecast_events = gpd.GeoDataFrame(
    forecast_df.copy(),
    geometry=gpd.points_from_xy(forecast_df["longitude"], forecast_df["latitude"]),
    crs="EPSG:4326",
).to_crs(PROJECTED_CRS)
faults_proj_check = faults.to_crs(PROJECTED_CRS)

joined_forecast = gpd.sjoin_nearest(
    forecast_events, faults_proj_check[["fault_segment_id", "geometry"]],
    max_distance=MAX_ASSIGN_DIST_KM * 1000, distance_col="dist_m",
)

parkfield_id = faults[faults["section_na"] == "Parkfield section"]["fault_segment_id"].iloc[0]
parkfield_events = joined_forecast[joined_forecast["fault_segment_id"] == parkfield_id]
print(parkfield_events[["latitude", "longitude", "magnitude", "dist_m"]])

       latitude   longitude  magnitude       dist_m
id                                                 
3062  35.777162 -120.327933        4.4   405.942067
3095  35.988763 -120.393604        4.4  9256.447123
3082  35.942845 -120.500985        4.2   979.608348
3106  35.881437 -120.490250        4.1  3814.008176
3075  35.950866 -120.479389        4.2   928.274626
3073  35.946227 -120.501318        4.0   706.567997
3069  35.949604 -120.500576        4.4   388.443903
3060  35.775555 -120.331222        4.4   749.614385
3063  35.883736 -120.428537        4.7   259.405535
3069  35.949833 -120.491619        4.2    58.753953
3069  35.951087 -120.495404        4.2     3.811708
3066  35.944809 -120.481333        4.0   362.755465


In [15]:
import os
etas_baseline_forecast.to_parquet("../data/processed/etas_baseline_forecast.parquet", index=False)
print("Saved data/processed/etas_baseline_forecast.parquet")


Saved data/processed/etas_baseline_forecast.parquet
